In [11]:
import numpy as np
import pandas as pd
import xarray as xr

rng = np.random.default_rng(0)

ndvi = xr.DataArray(
    rng.uniform(0.1, 0.9, size=(4, 3, 5)).round(2),
    dims=("time", "y", "x"),
    coords={
        "time": pd.to_datetime(["2023-06-01", "2023-07-01", "2023-08-01", "2023-09-01"]),
        "y": [30, 20, 10],
        "x": [100, 110, 120, 130, 140],
    },
    name="ndvi",
)
ndvi

<xarray.DataArray 'ndvi' (time: 4, y: 3, x: 5)> Size: 480B
array([[[0.61, 0.32, 0.13, 0.11, 0.75],
        [0.83, 0.59, 0.68, 0.53, 0.85],
        [0.75, 0.1 , 0.79, 0.13, 0.68]],

       [[0.24, 0.79, 0.53, 0.34, 0.44],
        [0.12, 0.2 , 0.64, 0.62, 0.59],
        [0.41, 0.9 , 0.88, 0.65, 0.62]],

       [[0.65, 0.41, 0.21, 0.68, 0.52],
        [0.35, 0.49, 0.81, 0.85, 0.39],
        [0.56, 0.36, 0.58, 0.37, 0.41]],

       [[0.81, 0.28, 0.6 , 0.17, 0.77],
        [0.73, 0.29, 0.8 , 0.15, 0.37],
        [0.22, 0.46, 0.74, 0.28, 0.14]]])
Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
  * y        (y) int64 24B 30 20 10
  * x        (x) int64 40B 100 110 120 130 140

In [13]:
ndvi.coords

Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
  * y        (y) int64 24B 30 20 10
  * x        (x) int64 40B 100 110 120 130 140

In [14]:
ndvi.x

<xarray.DataArray 'x' (x: 5)> Size: 40B
array([100, 110, 120, 130, 140])
Coordinates:
  * x        (x) int64 40B 100 110 120 130 140

In [15]:
ndvi.dims

('time', 'y', 'x')

In [16]:
ndvi.shape

(4, 3, 5)

In [17]:
ndvi.isel(time=2, y=1, x=3)

<xarray.DataArray 'ndvi' ()> Size: 8B
array(0.85)
Coordinates:
    time     datetime64[us] 8B 2023-08-01
    y        int64 8B 20
    x        int64 8B 130

In [18]:
ndvi.sel(x=123, y=18, method="nearest")

<xarray.DataArray 'ndvi' (time: 4)> Size: 32B
array([0.68, 0.64, 0.81, 0.8 ])
Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
    y        int64 8B 20
    x        int64 8B 120

In [19]:
ndvi.sel(x=slice(110, 130), y= slice(10, 20))

<xarray.DataArray 'ndvi' (time: 4, y: 0, x: 3)> Size: 0B
array([], shape=(4, 0, 3), dtype=float64)
Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
  * y        (y) int64 0B 
  * x        (x) int64 24B 110 120 130

In [21]:
ndvi.mean('time')

<xarray.DataArray 'ndvi' (y: 3, x: 5)> Size: 120B
array([[0.5775, 0.45  , 0.3675, 0.325 , 0.62  ],
       [0.5075, 0.3925, 0.7325, 0.5375, 0.55  ],
       [0.485 , 0.455 , 0.7475, 0.3575, 0.4625]])
Coordinates:
  * y        (y) int64 24B 30 20 10
  * x        (x) int64 40B 100 110 120 130 140

In [22]:
ndvi.mean(["y", "x"])

<xarray.DataArray 'ndvi' (time: 4)> Size: 32B
array([0.52333333, 0.53133333, 0.50933333, 0.454     ])
Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01

In [24]:
ndvi.max()

<xarray.DataArray 'ndvi' ()> Size: 8B
array(0.9)

In [28]:
masked=ndvi.where(ndvi >= 0.3)

In [30]:
valid_count = masked.notnull().sum('time')
valid_count

<xarray.DataArray 'ndvi' (y: 3, x: 5)> Size: 120B
array([[3, 3, 2, 2, 4],
       [3, 2, 4, 3, 4],
       [3, 3, 4, 2, 3]])
Coordinates:
  * y        (y) int64 24B 30 20 10
  * x        (x) int64 40B 100 110 120 130 140

In [31]:
with_skip = masked.mean("time", skipna=True)
without_skip = masked.mean("time", skipna=False)

In [34]:
a = ndvi.isel(x=slice(0, 3))     # x = 100, 110, 120
b = ndvi.isel(x=slice(2, 5))     # x = 120, 130, 140

diff_xarray = a - b
diff_numpy = a.values - b.values

print(diff_xarray.shape)   # (4, 3, 1)
print(diff_numpy.shape)    # (4, 3, 3)

(4, 3, 1)
(4, 3, 3)


In [35]:
bsi = xr.DataArray(
    rng.uniform(-0.3, 0.3, size=ndvi.shape).round(2),
    dims=ndvi.dims,
    coords=ndvi.coords,
    name="bsi",
)

ds = xr.Dataset({"ndvi": ndvi, "bsi": bsi})
ds

<xarray.Dataset> Size: 1kB
Dimensions:  (time: 4, y: 3, x: 5)
Coordinates:
  * time     (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
  * y        (y) int64 24B 30 20 10
  * x        (x) int64 40B 100 110 120 130 140
Data variables:
    ndvi     (time, y, x) float64 480B 0.61 0.32 0.13 0.11 ... 0.74 0.28 0.14
    bsi      (time, y, x) float64 480B -0.06 -0.18 -0.25 ... -0.26 0.2 -0.26

In [36]:
ds["difference"] = ds["ndvi"] - ds["bsi"]
ds

<xarray.Dataset> Size: 2kB
Dimensions:     (time: 4, y: 3, x: 5)
Coordinates:
  * time        (time) datetime64[us] 32B 2023-06-01 2023-07-01 ... 2023-09-01
  * y           (y) int64 24B 30 20 10
  * x           (x) int64 40B 100 110 120 130 140
Data variables:
    ndvi        (time, y, x) float64 480B 0.61 0.32 0.13 0.11 ... 0.74 0.28 0.14
    bsi         (time, y, x) float64 480B -0.06 -0.18 -0.25 ... -0.26 0.2 -0.26
    difference  (time, y, x) float64 480B 0.67 0.5 0.38 0.06 ... 1.0 0.08 0.4